# BERT: Under the Hood (Deep Dive)

**"Is BERT an Algorithm?"**

Strictly speaking, **No**. 
*   **The Algorithm** is the **Transformer** (specifically the Encoder part), which uses mechanisms like *Self-Attention* and *Backpropagation*.
*   **BERT** is a **Model Architecture** (a specific configuration of that algorithm) that has been **Pre-trained** on a massive amount of text.

Think of it this way:
*   **Algorithm (Engine):** Internal Combustion Engine.
*   **Model (Car):** Ferrari 488 GTB.
*   **BERT:** A specific Ferrari engine that has already been driven 1 million miles and knows exactly how to handle every curve.

This notebook will peel back the layers and show you the raw numbers and attention maps that make BERT work.


In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel, BertForMaskedLM

# Setup for visualization
sns.set(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 8)

print("Libraries loaded. Ready to explore!")


## Step 1: The Tokenizer (Words to Numbers)

BERT doesn't read strings; it reads numbers (IDs).
It uses a **WordPiece** tokenizer. It splits words into sub-words if it doesn't recognize them.


In [ ]:
# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

text = "The quick brown fox jumps over the lazy dog."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text: {text}")
print(f"Tokens:       {tokens}")
print(f"Token IDs:    {token_ids}")

# Special Tokens
# [CLS] = Classification (Start of sentence)
# [SEP] = Separator (End of sentence)
encoded_input = tokenizer(text, return_tensors='pt')
print(f"\nWith Special Tokens: {encoded_input['input_ids']}")


## Step 2: Embeddings & The Encoder

Once we have IDs, BERT looks up their **Embeddings**.
*   **Token Embeddings:** What the word means.
*   **Position Embeddings:** Where the word is in the sentence.
*   **Segment Embeddings:** Which sentence it belongs to (Sentence A or B).

These are summed up and fed into the **Encoder Stack** (12 layers for BERT-Base, 24 for Large).


In [ ]:
# Load pre-trained model (weights)
model = BertModel.from_pretrained('bert-base-uncased', output_attentions=True)

# Forward pass (feeding the text into the model)
with torch.no_grad():
    outputs = model(**encoded_input)

# The 'last_hidden_state' is the final understanding of each word after passing through all 12 layers.
last_hidden_states = outputs.last_hidden_state
print(f"Shape of Output: {last_hidden_states.shape}") 
print("(Batch Size, Sequence Length, Hidden Dimension)")
print("\nThis means for each of our token, BERT output a vector of size 768 representing its meaning in context.")


## Step 3: Visualizing Attention (The "Brain")

This is the magic. **Self-Attention** allows every word to "look at" every other word to understand context.
Let's visualize the attention map for a specific layer/head.


In [ ]:
# Get attention from the outputs
attentions = outputs.attentions # List of 12 tensors (one per layer)

# Let's look at Layer 11, Head 1
layer = 10
head = 0
attention_matrix = attentions[layer][0, head].numpy()

tokens_with_special = tokenizer.convert_ids_to_tokens(encoded_input['input_ids'][0])

plt.figure(figsize=(12, 10))
sns.heatmap(attention_matrix, xticklabels=tokens_with_special, yticklabels=tokens_with_special, cmap="viridis")
plt.title(f"Attention Map (Layer {layer+1}, Head {head+1})")
plt.xlabel("Key (Looking at)")
plt.ylabel("Query (Word doing the looking)")
plt.show()

print("Darker/Brighter colors mean the word on the Y-axis is paying more attention to the word on the X-axis.")


## Step 4: Masked Language Modeling (The Prediction)

Finally, let's see the probabilities for predicting a masked word.


In [ ]:
# Load the model specifically with a Language Modeling head
mlm_model = BertForMaskedLM.from_pretrained('bert-base-uncased')

masked_text = "The doctor ran to the [MASK]."
inputs = tokenizer(masked_text, return_tensors="pt")

with torch.no_grad():
    outputs = mlm_model(**inputs)
    predictions = outputs.logits

# Get the index of the [MASK] token
mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]

# Get the top 5 predicted tokens
predicted_token_id = predictions[0, mask_token_index].topk(5).indices.tolist()[0]
predicted_probs = predictions[0, mask_token_index].softmax(dim=0).topk(5).values.tolist()[0]

print(f"Sentence: {masked_text}")
print("Predictions:")
for id, prob in zip(predicted_token_id, predicted_probs):
    token = tokenizer.decode([id])
    print(f"{token:<10} | {prob*100:.2f}%")
